# J1 League – Player Value Dataset Builder
Merges StatsBomb event data with Transfermarkt market values to produce a modelling-ready player table.

In [1]:
import json
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1  Load Data

In [2]:
# ── Transfermarkt ──────────────────────────────────────────────────────────
tm = pd.read_csv('j1_league_player_values.csv')
print(f'Transfermarkt rows: {len(tm):,}')
tm.head()

Transfermarkt rows: 698


,player,team,market_value_eur,market_value_raw
0,Tomoki Hayakawa,Kashima Antlers,"1,500,000.00",€1.50m
1,Taiki Yamada,Kashima Antlers,"100,000.00",€100k
2,Yuji Kajikawa,Kashima Antlers,"100,000.00",€100k
3,Haruto Fujii,Kashima Antlers,NaN,-
4,Ikuma Sekigawa,Kashima Antlers,"1,200,000.00",€1.20m


In [3]:
# ── StatsBomb events ───────────────────────────────────────────────────────
# The JSON is already flat (dot-notation keys, one dict per event).
print('Loading sb_events.json …')
with open('sb_events.json', 'r', encoding='utf-8') as f:
    raw = json.load(f)

sb = pd.DataFrame(raw)
print(f'StatsBomb events: {len(sb):,}  |  matches: {sb["match_id"].nunique()}')

Loading sb_events.json …


StatsBomb events: 1,244,341  |  matches: 376


## 2  Inspect & Clean Transfermarkt Data

In [4]:
print(tm.dtypes)
print(f'\nMissing market_value_eur: {tm["market_value_eur"].isna().sum()}')

player               object
team                 object
market_value_eur    float64
market_value_raw     object
dtype: object

Missing market_value_eur: 106


In [5]:
def clean_name(name: str) -> str:
    """Lowercase, strip whitespace, remove accents, collapse spaces."""
    if not isinstance(name, str):
        return ''
    name = unicodedata.normalize('NFKD', name)
    name = ''.join(c for c in name if not unicodedata.combining(c))
    name = name.lower().strip()
    name = re.sub(r'\s+', ' ', name)
    return name


tm['player_name_clean'] = tm['player'].apply(clean_name)
tm['team_name_clean']   = tm['team'].apply(clean_name)

# Ensure numeric and drop rows without a market value
tm['market_value_eur'] = pd.to_numeric(tm['market_value_eur'], errors='coerce')
tm = tm.dropna(subset=['market_value_eur']).reset_index(drop=True)

print(f'Transfermarkt rows after dropping missing values: {len(tm):,}')
print(tm[['player', 'team', 'player_name_clean', 'team_name_clean', 'market_value_eur']].head(10))

Transfermarkt rows after dropping missing values: 592
            player             team player_name_clean  team_name_clean  \
0  Tomoki Hayakawa  Kashima Antlers   tomoki hayakawa  kashima antlers   
1     Taiki Yamada  Kashima Antlers      taiki yamada  kashima antlers   
2    Yuji Kajikawa  Kashima Antlers     yuji kajikawa  kashima antlers   
3   Ikuma Sekigawa  Kashima Antlers    ikuma sekigawa  kashima antlers   
4    Naomichi Ueda  Kashima Antlers     naomichi ueda  kashima antlers   
5    Tae-hyeon Kim  Kashima Antlers     tae-hyeon kim  kashima antlers   
6      Kaito Chida  Kashima Antlers       kaito chida  kashima antlers   
7   Keisuke Tsukui  Kashima Antlers    keisuke tsukui  kashima antlers   
8      Ryoya Ogawa  Kashima Antlers       ryoya ogawa  kashima antlers   
9       Koki Anzai  Kashima Antlers        koki anzai  kashima antlers   

   market_value_eur  
0      1,500,000.00  
1        100,000.00  
2        100,000.00  
3      1,200,000.00  
4      1,200,000.00  

## 3  Build Player-Level StatsBomb Performance Table

### 3a  Minutes Played
We derive minutes from `Starting XI` and `Substitution` events.
- Starting player not subbed off → plays until the last event minute of the match.
- Starting player subbed off at minute M → plays M minutes.
- Substitute entering at minute M → plays `match_end_minute − M` minutes.

In [6]:
def compute_minutes_played(sb: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame with (player_id, player_name, team_name, match_id, minutes)."""
    records = []

    # Max minute played per match (used as the match end time)
    match_end = sb.groupby('match_id')['minute'].max().to_dict()

    # ── Starting lineups ──────────────────────────────────────────────────
    xi_events = sb[sb['type.name'] == 'Starting XI'].copy()
    for _, row in xi_events.iterrows():
        lineup = row.get('tactics.lineup')
        if not isinstance(lineup, list):
            continue
        match_id  = row['match_id']
        team_name = row['team.name']
        for player in lineup:
            records.append({
                'match_id':    match_id,
                'player_id':   player['player.id'],
                'player_name': player['player.name'],
                'team_name':   team_name,
                'start_min':   0,
                'end_min':     match_end.get(match_id, 90),
                'is_starter':  True,
            })

    minutes_df = pd.DataFrame(records)

    # ── Substitutions ─────────────────────────────────────────────────────
    sub_events = sb[sb['type.name'] == 'Substitution'].copy()
    for _, row in sub_events.iterrows():
        match_id     = row['match_id']
        team_name    = row['team.name']
        sub_minute   = row['minute']
        player_out_id   = row['player.id']
        player_in_id    = row.get('substitution.replacement.id')
        player_in_name  = row.get('substitution.replacement.name')

        # Cap the player who was subbed off
        mask = (
            (minutes_df['match_id']   == match_id) &
            (minutes_df['player_id']  == player_out_id)
        )
        minutes_df.loc[mask, 'end_min'] = sub_minute

        # Add the substitute
        if pd.notna(player_in_id):
            minutes_df = pd.concat([
                minutes_df,
                pd.DataFrame([{
                    'match_id':    match_id,
                    'player_id':   int(player_in_id),
                    'player_name': player_in_name,
                    'team_name':   team_name,
                    'start_min':   sub_minute,
                    'end_min':     match_end.get(match_id, 90),
                    'is_starter':  False,
                }])
            ], ignore_index=True)

    minutes_df['minutes'] = (minutes_df['end_min'] - minutes_df['start_min']).clip(lower=0)
    return minutes_df[['player_id', 'player_name', 'team_name', 'match_id', 'minutes']]


minutes_per_match = compute_minutes_played(sb)

# Aggregate to season totals
minutes_season = (
    minutes_per_match
    .groupby(['player_id', 'player_name', 'team_name'], as_index=False)
    ['minutes'].sum()
    .rename(columns={'minutes': 'minutes_played'})
)
print(f'Players with minutes data: {len(minutes_season):,}')
minutes_season.head()

Players with minutes data: 608


,player_id,player_name,team_name,minutes_played
0,3175,Eiji Kawashima,Júbilo Iwata,2988
1,3530,Hiroki Sakai,Urawa Reds,615
2,3743,Amadou Bakayoko,Consadole Sapporo,140
3,5679,Yūya Ōsako,Vissel Kobe,3055
4,5682,Gen Shōji,FC Machida Zelvia,2854


### 3b  xA (Expected Assists)
Join assist passes to their shot's xG via `pass.assisted_shot_id` ↔ shot `id`.

In [7]:
# Build shot-id → xG lookup
shots = sb[sb['type.name'] == 'Shot'][['id', 'shot.statsbomb_xg']].dropna(subset=['shot.statsbomb_xg'])
shot_xg = shots.set_index('id')['shot.statsbomb_xg'].to_dict()

# Passes that preceded a shot
assist_passes = sb[
    (sb['type.name'] == 'Pass') &
    (sb['pass.shot_assist'].fillna(False) | sb['pass.goal_assist'].fillna(False))
].copy()

assist_passes['xA'] = assist_passes['pass.assisted_shot_id'].map(shot_xg).fillna(0)

xa_player = (
    assist_passes
    .groupby(['player.id', 'team.name'], as_index=False)
    ['xA'].sum()
    .rename(columns={'player.id': 'player_id', 'team.name': 'team_name_xa'})
)
print(f'Players with xA data: {len(xa_player):,}')

C:\Users\User\AppData\Local\Temp\ipykernel_32352\4054726148.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (sb['pass.shot_assist'].fillna(False) | sb['pass.goal_assist'].fillna(False))


Players with xA data: 516


### 3c  Aggregate All Performance Metrics

In [8]:
def aggregate_events(sb: pd.DataFrame) -> pd.DataFrame:
    """Return one row per player with summed season metrics."""
    # Only events with a player attached
    df = sb[sb['player.id'].notna()].copy()

    # ── Shots & goals ─────────────────────────────────────────────────────
    shot_df = df[df['type.name'] == 'Shot'].copy()
    shot_df['is_goal'] = (shot_df['shot.outcome.name'] == 'Goal').astype(int)
    shot_agg = shot_df.groupby(['player.id', 'team.name']).agg(
        shots          = ('id', 'count'),
        goals          = ('is_goal', 'sum'),
        xG             = ('shot.statsbomb_xg', 'sum'),
    ).reset_index()

    # ── Passes ────────────────────────────────────────────────────────────
    pass_df = df[df['type.name'] == 'Pass'].copy()
    pass_df['completed']    = pass_df['pass.outcome.name'].isna().astype(int)  # null = success
    pass_df['key_pass']     = pass_df['pass.shot_assist'].fillna(False).astype(int)
    pass_df['assist']       = pass_df['pass.goal_assist'].fillna(False).astype(int)
    # Progressive: end x at least 10m (StatsBomb units) further toward goal than start x
    start_x = pass_df['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else np.nan)
    end_x   = pass_df['pass.end_location'].apply(lambda loc: loc[0] if isinstance(loc, list) else np.nan)
    pass_df['progressive']  = ((end_x - start_x) >= 10).astype(int)

    pass_agg = pass_df.groupby(['player.id', 'team.name']).agg(
        passes             = ('id', 'count'),
        completed_passes   = ('completed', 'sum'),
        key_passes         = ('key_pass', 'sum'),
        assists            = ('assist', 'sum'),
        progressive_passes = ('progressive', 'sum'),
    ).reset_index()

    # ── Pressures ─────────────────────────────────────────────────────────
    pressure_agg = (
        df[df['type.name'] == 'Pressure']
        .groupby(['player.id', 'team.name'])
        .agg(pressures=('id', 'count'))
        .reset_index()
    )

    # ── Interceptions ─────────────────────────────────────────────────────
    intercept_agg = (
        df[df['type.name'] == 'Interception']
        .groupby(['player.id', 'team.name'])
        .agg(interceptions=('id', 'count'))
        .reset_index()
    )

    # ── Tackles (Duel – Tackle type) ──────────────────────────────────────
    tackle_agg = (
        df[(df['type.name'] == 'Duel') & (df['duel.type.name'] == 'Tackle')]
        .groupby(['player.id', 'team.name'])
        .agg(tackles=('id', 'count'))
        .reset_index()
    )

    # ── Dribbles ──────────────────────────────────────────────────────────
    dribble_agg = (
        df[df['type.name'] == 'Dribble']
        .groupby(['player.id', 'team.name'])
        .agg(
            dribbles           = ('id', 'count'),
            successful_dribbles = ('dribble.outcome.name', lambda x: (x == 'Complete').sum()),
        )
        .reset_index()
    )

    # ── Merge all metrics together ─────────────────────────────────────────
    base = shot_agg.rename(columns={'player.id': 'player_id', 'team.name': 'team_name'})
    for agg_df in [pass_agg, pressure_agg, intercept_agg, tackle_agg, dribble_agg]:
        agg_df = agg_df.rename(columns={'player.id': 'player_id', 'team.name': 'team_name'})
        base = base.merge(agg_df, on=['player_id', 'team_name'], how='outer')

    return base.fillna(0)


perf = aggregate_events(sb)
print(f'Players in performance table: {len(perf):,}')
perf.head()

C:\Users\User\AppData\Local\Temp\ipykernel_32352\1730100682.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pass_df['key_pass']     = pass_df['pass.shot_assist'].fillna(False).astype(int)
C:\Users\User\AppData\Local\Temp\ipykernel_32352\1730100682.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pass_df['assist']       = pass_df['pass.goal_assist'].fillna(False).astype(int)


Players in performance table: 608


,player_id,team_name,shots,goals,xG,passes,completed_passes,key_passes,assists,progressive_passes,pressures,interceptions,tackles,dribbles,successful_dribbles
0,"3,175.00",Júbilo Iwata,1.00,0.00,0.06,946.00,601.00,2.00,0.00,640.00,1.00,0.00,0.00,0.00,0.00
1,"3,530.00",Urawa Reds,4.00,1.00,0.32,421.00,327.00,4.00,2.00,142.00,64.00,9.00,13.00,10.00,4.00
2,"3,743.00",Consadole Sapporo,3.00,1.00,1.12,41.00,23.00,3.00,0.00,5.00,16.00,1.00,1.00,2.00,1.00
3,"5,679.00",Vissel Kobe,89.00,11.00,13.07,947.00,636.00,49.00,7.00,317.00,464.00,3.00,14.00,32.00,12.00
4,"5,682.00",FC Machida Zelvia,13.00,1.00,1.24,"1,169.00",918.00,7.00,0.00,592.00,217.00,19.00,37.00,7.00,6.00


### 3d  Combine Minutes + xA + Performance

In [9]:
# Merge minutes onto performance
sb_players = minutes_season.merge(
    perf,
    on=['player_id', 'team_name'],
    how='left',
).fillna(0)

# Merge xA
sb_players = sb_players.merge(
    xa_player.rename(columns={'team_name_xa': 'team_name'}),
    on=['player_id', 'team_name'],
    how='left',
).fillna(0)

# ── Per-90 stats ──────────────────────────────────────────────────────────
per90_cols = ['goals', 'shots', 'xG', 'xA', 'key_passes', 'assists',
              'passes', 'completed_passes', 'progressive_passes',
              'pressures', 'interceptions', 'tackles',
              'dribbles', 'successful_dribbles']

nineties = sb_players['minutes_played'] / 90
for col in per90_cols:
    if col in sb_players.columns:
        sb_players[f'{col}_per90'] = sb_players[col] / nineties.replace(0, np.nan)

# Pass completion rate
sb_players['pass_completion_pct'] = (
    sb_players['completed_passes'] / sb_players['passes'].replace(0, np.nan) * 100
)

print(f'StatsBomb player table shape: {sb_players.shape}')
sb_players.head()

StatsBomb player table shape: (608, 33)


,player_id,player_name,team_name,minutes_played,shots,goals,xG,passes,completed_passes,key_passes,assists,progressive_passes,pressures,interceptions,tackles,dribbles,successful_dribbles,xA,goals_per90,shots_per90,xG_per90,xA_per90,key_passes_per90,assists_per90,passes_per90,completed_passes_per90,progressive_passes_per90,pressures_per90,interceptions_per90,tackles_per90,dribbles_per90,successful_dribbles_per90,pass_completion_pct
0,3175,Eiji Kawashima,Júbilo Iwata,2988,1.00,0.00,0.06,946.00,601.00,2.00,0.00,640.00,1.00,0.00,0.00,0.00,0.00,0.09,0.00,0.03,0.00,0.00,0.06,0.00,28.49,18.10,19.28,0.03,0.00,0.00,0.00,0.00,63.53
1,3530,Hiroki Sakai,Urawa Reds,615,4.00,1.00,0.32,421.00,327.00,4.00,2.00,142.00,64.00,9.00,13.00,10.00,4.00,0.38,0.15,0.59,0.05,0.06,0.59,0.29,61.61,47.85,20.78,9.37,1.32,1.90,1.46,0.59,77.67
2,3743,Amadou Bakayoko,Consadole Sapporo,140,3.00,1.00,1.12,41.00,23.00,3.00,0.00,5.00,16.00,1.00,1.00,2.00,1.00,0.10,0.64,1.93,0.72,0.06,1.93,0.00,26.36,14.79,3.21,10.29,0.64,0.64,1.29,0.64,56.10
3,5679,Yūya Ōsako,Vissel Kobe,3055,89.00,11.00,13.07,947.00,636.00,49.00,7.00,317.00,464.00,3.00,14.00,32.00,12.00,5.85,0.32,2.62,0.39,0.17,1.44,0.21,27.90,18.74,9.34,13.67,0.09,0.41,0.94,0.35,67.16
4,5682,Gen Shōji,FC Machida Zelvia,2854,13.00,1.00,1.24,"1,169.00",918.00,7.00,0.00,592.00,217.00,19.00,37.00,7.00,6.00,0.33,0.03,0.41,0.04,0.01,0.22,0.00,36.86,28.95,18.67,6.84,0.60,1.17,0.22,0.19,78.53


### 3e  Clean Names for Joining

In [10]:
sb_players['player_name_clean'] = sb_players['player_name'].apply(clean_name)
sb_players['team_name_clean']   = sb_players['team_name'].apply(clean_name)

# Preview the clean keys
sb_players[['player_name', 'team_name', 'player_name_clean', 'team_name_clean']].head(10)

,player_name,team_name,player_name_clean,team_name_clean
0,Eiji Kawashima,Júbilo Iwata,eiji kawashima,jubilo iwata
1,Hiroki Sakai,Urawa Reds,hiroki sakai,urawa reds
2,Amadou Bakayoko,Consadole Sapporo,amadou bakayoko,consadole sapporo
3,Yūya Ōsako,Vissel Kobe,yuya osako,vissel kobe
4,Gen Shōji,FC Machida Zelvia,gen shoji,fc machida zelvia
5,Shinji Kagawa,Cerezo Osaka,shinji kagawa,cerezo osaka
6,Yuto Nagatomo,Tokyo,yuto nagatomo,tokyo
7,Gaku Shibasaki,Kashima Antlers,gaku shibasaki,kashima antlers
8,Hotaru Yamaguchi,Vissel Kobe,hotaru yamaguchi,vissel kobe
9,Genki Haraguchi,Urawa Reds,genki haraguchi,urawa reds


## 4  Merge Transfermarkt Values onto StatsBomb Table

In [11]:
# Detect duplicates on join keys before merging
tm_dupes   = tm.duplicated(subset=['player_name_clean', 'team_name_clean'], keep=False).sum()
sb_dupes   = sb_players.duplicated(subset=['player_name_clean', 'team_name_clean'], keep=False).sum()
print(f'Transfermarkt duplicate join-key rows : {tm_dupes}')
print(f'StatsBomb duplicate join-key rows     : {sb_dupes}')

# If StatsBomb has duplicates (player at same club after mid-season move), keep row with most minutes
sb_players = (
    sb_players
    .sort_values('minutes_played', ascending=False)
    .drop_duplicates(subset=['player_name_clean', 'team_name_clean'], keep='first')
    .reset_index(drop=True)
)

# Keep only the most valuable row per player in TM (edge case for duplicate scraped rows)
tm_clean = (
    tm
    .sort_values('market_value_eur', ascending=False)
    .drop_duplicates(subset=['player_name_clean', 'team_name_clean'], keep='first')
    .reset_index(drop=True)
)

tm_cols = ['player_name_clean', 'team_name_clean', 'market_value_eur', 'market_value_raw']
merged = sb_players.merge(
    tm_clean[tm_cols],
    on=['player_name_clean', 'team_name_clean'],
    how='inner',
)

print(f'\nStatsBomb players : {len(sb_players):,}')
print(f'Transfermarkt players : {len(tm_clean):,}')
print(f'Inner-join matches: {len(merged):,}')

Transfermarkt duplicate join-key rows : 2
StatsBomb duplicate join-key rows     : 0

StatsBomb players : 608
Transfermarkt players : 591
Inner-join matches: 146


### 4a  Diagnose Unmatched Players
Players present in StatsBomb but missing from Transfermarkt after the join — useful for detecting name mismatches.

In [12]:
unmatched = sb_players[
    ~sb_players['player_name_clean'].isin(merged['player_name_clean'])
][['player_name', 'team_name', 'player_name_clean', 'team_name_clean', 'minutes_played']]

print(f'StatsBomb players not matched in TM: {len(unmatched):,}')
print('\nTop unmatched by minutes played:')
print(unmatched.sort_values('minutes_played', ascending=False).head(20).to_string(index=False))

StatsBomb players not matched in TM: 454

Top unmatched by minutes played:
                         player_name           team_name                    player_name_clean     team_name_clean  minutes_played
Matheus Caldeira Vidotto de Oliveira         Tokyo Verdy matheus caldeira vidotto de oliveira         tokyo verdy            3664
                      Ryoma Watanabe          Urawa Reds                       ryoma watanabe          urawa reds            3580
                         Il-Gyu Park          Sagan Tosu                          il-gyu park          sagan tosu            3576
                          Kosei Tani   FC Machida Zelvia                           kosei tani   fc machida zelvia            3565
                       Soya Fujiwara     Albirex Niigata                        soya fujiwara     albirex niigata            3546
                    Keisuke Kurokawa         Gamba Osaka                     keisuke kurokawa         gamba osaka            3529
               

## 5  Apply Filters & Finalise

In [13]:
MIN_MINUTES = 600  # roughly 7 full matches

final = merged[
    (merged['minutes_played'] >= MIN_MINUTES) &
    (merged['market_value_eur'].notna()) &
    (merged['market_value_eur'] > 0)
].copy().reset_index(drop=True)

final['log_market_value'] = np.log(final['market_value_eur'])

# Select and order final columns
id_cols        = ['player_id', 'player_name', 'team_name']
total_cols     = ['minutes_played', 'goals', 'assists', 'shots', 'xG', 'xA',
                  'key_passes', 'passes', 'completed_passes', 'progressive_passes',
                  'pressures', 'interceptions', 'tackles', 'dribbles', 'successful_dribbles',
                  'pass_completion_pct']
per90_out_cols = [c for c in final.columns if c.endswith('_per90')]
target_cols    = ['market_value_eur', 'market_value_raw', 'log_market_value']

keep_cols = id_cols + total_cols + per90_out_cols + target_cols
final = final[[c for c in keep_cols if c in final.columns]]

print(f'Final dataset shape: {final.shape}')
final.head()

Final dataset shape: (105, 36)


,player_id,player_name,team_name,minutes_played,goals,assists,shots,xG,xA,key_passes,passes,completed_passes,progressive_passes,pressures,interceptions,tackles,dribbles,successful_dribbles,pass_completion_pct,goals_per90,shots_per90,xG_per90,xA_per90,key_passes_per90,assists_per90,passes_per90,completed_passes_per90,progressive_passes_per90,pressures_per90,interceptions_per90,tackles_per90,dribbles_per90,successful_dribbles_per90,market_value_eur,market_value_raw,log_market_value
0,38523,Jin-Hyeon Kim,Cerezo Osaka,3677,0.00,0.00,1.00,0.05,0.17,4.00,"1,344.00",846.00,"1,032.00",2.00,0.00,1.00,2.00,2.00,62.95,0.00,0.02,0.00,0.00,0.10,0.00,32.90,20.71,25.26,0.05,0.00,0.02,0.05,0.05,"100,000.00",€100k,11.51
1,37514,Jun Ichimori,Gamba Osaka,3671,0.00,0.00,0.00,0.00,0.00,0.00,"1,495.00","1,019.00",994.00,2.00,0.00,0.00,2.00,2.00,68.16,0.00,0.00,0.00,0.00,0.00,0.00,36.65,24.98,24.37,0.05,0.00,0.00,0.05,0.05,"300,000.00",€300k,12.61
2,37121,Shinnosuke Nakatani,Gamba Osaka,3671,4.00,0.00,21.00,3.12,0.80,7.00,"2,244.00","1,945.00",955.00,260.00,29.00,36.00,7.00,7.00,86.68,0.10,0.51,0.08,0.02,0.17,0.00,55.01,47.68,23.41,6.37,0.71,0.88,0.17,0.17,"1,300,000.00",€1.30m,14.08
3,126936,Tomoki Hayakawa,Kashima Antlers,3656,0.00,0.00,0.00,0.00,0.09,1.00,"1,432.00",939.00,"1,123.00",3.00,0.00,0.00,2.00,2.00,65.57,0.00,0.00,0.00,0.00,0.02,0.00,35.25,23.12,27.64,0.07,0.00,0.00,0.05,0.05,"1,500,000.00",€1.50m,14.22
4,38272,Keisuke Osako,Sanfrecce Hiroshima,3649,0.00,0.00,0.00,0.00,0.01,1.00,"1,024.00",530.00,881.00,1.00,0.00,0.00,2.00,1.00,51.76,0.00,0.00,0.00,0.00,0.02,0.00,25.26,13.07,21.73,0.02,0.00,0.00,0.05,0.02,"2,000,000.00",€2.00m,14.51


## 6  Sanity Checks

In [14]:
print('=== Market value distribution (EUR) ===')
print(final['market_value_eur'].describe().apply('{:,.0f}'.format))

print('\n=== Minutes played distribution ===')
print(final['minutes_played'].describe().apply('{:,.1f}'.format))

print('\n=== Goals vs xG (season totals) ===')
print(f"Total goals : {final['goals'].sum():.0f}")
print(f"Total xG    : {final['xG'].sum():.1f}")

print('\n=== Players per team ===')
print(final['team_name'].value_counts().to_string())

=== Market value distribution (EUR) ===
count          105
mean       710,476
std        528,739
min         50,000
25%        350,000
50%        500,000
75%      1,000,000
max      3,000,000
Name: market_value_eur, dtype: object

=== Minutes played distribution ===
count      105.0
mean     2,165.1
std        960.0
min        610.0
25%      1,310.0
50%      2,077.0
75%      3,090.0
max      3,677.0
Name: minutes_played, dtype: object

=== Goals vs xG (season totals) ===
Total goals : 244
Total xG    : 233.8

=== Players per team ===
team_name
Nagoya Grampus         13
Kashima Antlers        12
Sanfrecce Hiroshima    11
Gamba Osaka            10
Tokyo Verdy            10
Vissel Kobe             9
Yokohama F. Marinos     9
Cerezo Osaka            8
Kashiwa Reysol          8
Kawasaki Frontale       8
Avispa Fukuoka          7


In [15]:
# Top 10 players by market value
print('Top 10 players by market value:')
print(
    final[['player_name', 'team_name', 'market_value_eur', 'goals', 'xG_per90', 'minutes_played']]
    .sort_values('market_value_eur', ascending=False)
    .head(10)
    .to_string(index=False)
)

Top 10 players by market value:
      player_name           team_name  market_value_eur  goals  xG_per90  minutes_played
     Hayao Kawabe Sanfrecce Hiroshima      3,000,000.00   0.00      0.07            1021
      Yuma Suzuki     Kashima Antlers      2,500,000.00  15.00      0.31            3236
    Keisuke Osako Sanfrecce Hiroshima      2,000,000.00   0.00      0.00            3649
       Mao Hosoya      Kashiwa Reysol      1,800,000.00   6.00      0.31            2561
     Hayato Araki Sanfrecce Hiroshima      1,800,000.00   3.00      0.08            2437
  Yasuto Wakizaka   Kawasaki Frontale      1,500,000.00   6.00      0.16            2883
Tetsushi Yamakawa         Vissel Kobe      1,500,000.00   3.00      0.03            3361
  Tomoki Hayakawa     Kashima Antlers      1,500,000.00   0.00      0.00            3656
     Shuto Nakano Sanfrecce Hiroshima      1,500,000.00   5.00      0.06            3547
       Taiyo Koga      Kashiwa Reysol      1,500,000.00   1.00      0.01      

## 7  Save Output

In [16]:
output_path = 'j1_player_value_statsbomb_merged.csv'
final.to_csv(output_path, index=False)
print(f'Saved {len(final):,} rows × {len(final.columns)} columns → {output_path}')

Saved 105 rows × 36 columns → j1_player_value_statsbomb_merged.csv
